In [26]:
!pip install groq -q
print("Groq installed successfully")

Groq installed successfully


In [27]:
import sqlite3
import pandas as pd
import os
from groq import Groq
import re
print("All libraries imported successfully")

All libraries imported successfully


In [28]:
import os
os.environ["GROQ_API_KEY"] = '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'
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "llama-3.1-8b-instant"
print("Using Model:", MODEL)

Using Model: llama-3.1-8b-instant


In [29]:
df = pd.read_csv('/content/drive/MyDrive/15 days intern dataset/student_performance.csv')
print("=== Data Loaded Successfully ===")

=== Data Loaded Successfully ===


In [30]:
print("No of rows:", len(df))
print("NO of columns:", len(df.columns))
print()

print("=== First 5 Rows ===")
print(df.head())

No of rows: 30
NO of columns: 13

=== First 5 Rows ===
   student_id          name  age  gender        department  semester  \
0        1001  Aarav Sharma   19    Male  Computer Science         2   
1        1002   Priya Patel   20  Female  Computer Science         2   
2        1003   Rohit Verma   19    Male       Electronics         2   
3        1004   Sneha Reddy   20  Female        Mechanical         2   
4        1005    Arjun Nair   19    Male  Computer Science         2   

   math_score  science_score  english_score  programming_score  \
0          85             78             72                 91   
1          76             82             88                 79   
2          65             74             61                 55   
3          70             80             75                 48   
4          92             88             81                 95   

   attendance_percentage       city  admission_year  
0                     92     Mumbai            2023  
1      

In [31]:
conn = sqlite3.connect('college.db')
df.to_sql('students', conn, if_exists = 'replace', index = False)
test_df = pd.read_sql_query("select name,age from students", conn)
print("=== Test Dataframe ===")
print(test_df)



=== Test Dataframe ===
              name  age
0     Aarav Sharma   19
1      Priya Patel   20
2      Rohit Verma   19
3      Sneha Reddy   20
4       Arjun Nair   19
5      Meera Joshi   20
6      Kiran Kumar   21
7      Divya Singh   19
8     Rahul Mishra   20
9       Ananya Das   19
10     Vikram Iyer   20
11     Pooja Gupta   19
12      Suresh Rao   21
13   Kavya Nambiar   20
14     Ajay Tiwari   19
15    Ritu Agarwal   20
16    Manoj Pandey   21
17  Swati Kulkarni   19
18  Deepak Chauhan   20
19    Nisha Kapoor   19
20   Harish Pillai   20
21     Tanvi Mehta   19
22    Sanjay Dubey   21
23   Preeti Saxena   20
24       Amit Bose   19
25      Rekha Nair   20
26   Gaurav Shukla   21
27   Sunita Pillai   19
28      Nitin Jain   20
29  Akanksha Yadav   19


In [32]:
conn = sqlite3.connect('college.db')
df.to_sql('students', conn, if_exists = 'replace', index = False)
test_df = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM STUDENTS", conn)
print("=== Test Dataframe ===")
print(test_df)



=== Test Dataframe ===
   total_rows
0          30


In [33]:
def get_schema(conn, table_name = "students"):
  """
  this is the description
  """

  cursor = conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns = cursor.fetchall()
  schema_lines = ["Table:", table_name]
  schema_lines.append("Columns:")

  for col in columns:
    schema_lines.append(f" - {col[1]} ({col[2]})")

  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")

  sammple_rows = cursor.fetchall()
  schema_lines.append("Sample Rows (first 3):")

  for row in sammple_rows:
    schema_lines.append(f" - {row}")

  return "\n".join(schema_lines)

schema = get_schema(conn)
print(schema)

Table:
students
Columns:
 - student_id (INTEGER)
 - name (TEXT)
 - age (INTEGER)
 - gender (TEXT)
 - department (TEXT)
 - semester (INTEGER)
 - math_score (INTEGER)
 - science_score (INTEGER)
 - english_score (INTEGER)
 - programming_score (INTEGER)
 - attendance_percentage (INTEGER)
 - city (TEXT)
 - admission_year (INTEGER)
Sample Rows (first 3):
 - (1001, 'Aarav Sharma', 19, 'Male', 'Computer Science', 2, 85, 78, 72, 91, 92, 'Mumbai', 2023)
 - (1002, 'Priya Patel', 20, 'Female', 'Computer Science', 2, 76, 82, 88, 79, 87, 'Ahmedabad', 2023)
 - (1003, 'Rohit Verma', 19, 'Male', 'Electronics', 2, 65, 74, 61, 55, 78, 'Delhi', 2023)


In [ ]:
def generate_sql(user_question,schema_text,client,model):
  """
  this is the description function for generating sql queries

  """

  system_prompt = f"""
You are an expert SQLite SQL assistant.

Generate only valid SQLite SQL queries.

Database Rules:
1. Use only the table named: students
2. Do not invent table names
3. Use only columns from the schema
4. Return only SQL query
5. No explanations
6. No markdown

Database Schema:
{schema_text}
"""

  response  = client.chat.completions.create(
    model = model,
    messages = [
      {"role": "system","content": system_prompt},
      {"role": "user","content": user_question}
    ],
    temperature = 0.0
  )

  sql_query = response.choices[0].message.content.strip()
  return sql_query

question = input("Enter the Question")
print("Question:", question)
print("Generating sql ...")

sql = generate_sql(question,schema,client,MODEL)
print("Generated SQL:", sql)


In [ ]:
def execute_sql(sql_query,conn):
    """
    Clean and execute SQL query on SQLite database.

    Parameters:
    sql_query : str
        SQL query generated by LLM

    conn : sqlite3.Connection
        SQLite database connection object

    Returns:
    result_df : pandas.DataFrame
        Query execution result
    """

    clean_sql = sql_query.strip()
    clean_sql = re.sub(r'```\S*', '', clean_sql)
    clean_sql = clean_sql.strip()

    try:
        result_df = pd.read_sql_query(clean_sql, conn)
        return result_df, None
    except Exception as e:
      return None, str(e)

print("Exceuting SQL:", sql)
result , error = execute_sql(sql,conn)

if error:
  print("Error:", error)
else:
  print("Query Returned:", len(result))
  print(result)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
#the complete text to sql agent function
def text_to_sql_agent(user_question,conn,client,model,verbose = True):
  """
    Complete AI Text-to-SQL Agent

    Workflow:
    1. Read database schema
    2. Generate SQL query using Groq LLM
    3. Execute SQL query
    4. Return results

    Parameters:
    user_question : str
        Natural language user query

    conn : sqlite3.Connection
        SQLite database connection

    client : Groq
        Groq API client

    model : str
        Groq model name

    verbose : bool
        Print intermediate outputs

    Returns:
    result_df : pandas.DataFrame
        Final query results
    """

  print("User Question:", user_question)
  print()

  if verbose:
    print("[Step 1] Reading Databse schema ...")
  schema_text = get_schema(conn)

  if verbose:
    print("Schema loaded successfully")
    print()

  if verbose:
    print("[Step 2]Generating SQL query with GROQ LLM........")

  generated_sql = generate_sql(user_question,schema_text,client,model)

  if verbose:
    print("Generated SQL:", generated_sql)
    print()
  if verbose:
    print("[Step 3] Executing SQL on the database.....")

  result_df,error = execute_sql(generated_sql,conn)

  if error:
    print("SQL Error:", error)
    return None, generated_sql
  else:
    print("Query Returned:", len(result_df))
    print()
    print("RESULTS:")
    print(result_df.to_string(index=False))


  return result_df, generated_sql

result,sql_used = text_to_sql_agent(question, conn, client, MODEL)
